# LeetCode #33 — Search in Rotated Sorted Array: Binary Search trên mảng bị xoay

**Mức độ:** Medium · **Chủ đề:** Array, Binary Search

Skeleton trên LeetCode:

```python
class Solution:
    def search(self, nums: List[int], target: int) -> int:
```

Mục tiêu của notebook:

1. Ôn lại **tìm kiếm nhị phân (Binary Search)** chuẩn: vì sao nó `O(log n)`, viết sao cho không lặp vô hạn.
2. Hiểu quan sát then chốt của mảng bị xoay: **cắt ở giữa thì luôn có ít nhất một nửa còn sắp xếp**.
3. Dùng nửa đã sắp xếp để quyết định **bỏ nửa nào**.
4. Cách giải thứ hai: **tìm điểm xoay** trước, rồi tìm kiếm nhị phân thường.

# 1. Bài toán là gì?

Một mảng `nums` ban đầu được sắp xếp **tăng dần**, các giá trị **đôi một khác nhau**. Trước khi đưa cho bạn, nó có thể đã bị **xoay** tại một chỉ số `k` chưa biết:

```text
[0, 1, 2, 4, 5, 6, 7]  xoay tại k = 3  ->  [4, 5, 6, 7, 0, 1, 2]
```

(Lấy `nums[k:]` đặt lên trước `nums[:k]`.)

Cho `nums` sau khi xoay và `target`. Trả về **chỉ số** của `target`, hoặc `-1` nếu không có. **Yêu cầu `O(log n)`.**

**Constraints (theo đề):**

- `1 <= nums.length <= 5000`
- `-10^4 <= nums[i], target <= 10^4`
- Các giá trị **khác nhau**; `nums` là một mảng tăng dần có thể đã bị xoay.

---

## Ví dụ

```text
nums = [4, 5, 6, 7, 0, 1, 2], target = 0  ->  4
nums = [4, 5, 6, 7, 0, 1, 2], target = 3  ->  -1
nums = [1], target = 0                    ->  -1
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Tìm kiếm nhị phân (binary search)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Tìm kiếm nhị phân (binary search)

### Ý tưởng

Trên mảng **đã sắp xếp**, so phần tử cần tìm với phần tử **ở giữa**: nếu nhỏ hơn thì chắc chắn nằm ở nửa trái, lớn hơn thì ở nửa phải. Mỗi bước loại **một nửa** → chỉ `O(log n)` bước (mảng 1 triệu phần tử: ~20 bước).

```text
tìm 23 trong [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
lo=0 hi=9 mid=4 → 16 < 23 → lo = 5
lo=5 hi=9 mid=7 → 56 > 23 → hi = 6
lo=5 hi=6 mid=5 → 23 = 23 → tìm thấy ở chỉ số 5
```

In [1]:
def tim_nhi_phan(a, x):
    lo, hi = 0, len(a) - 1                # khoảng tìm đóng [lo, hi]
    while lo <= hi:
        mid = (lo + hi) // 2
        print(f"   lo={lo} hi={hi} mid={mid} a[mid]={a[mid]}")
        if a[mid] == x:
            return mid
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

a_bs = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("→", tim_nhi_phan(a_bs, 23))
print("→", tim_nhi_phan(a_bs, 7), "(không có)")

   lo=0 hi=9 mid=4 a[mid]=16
   lo=5 hi=9 mid=7 a[mid]=56
   lo=5 hi=6 mid=5 a[mid]=23
→ 5
   lo=0 hi=9 mid=4 a[mid]=16
   lo=0 hi=3 mid=1 a[mid]=5
   lo=2 hi=3 mid=2 a[mid]=8
→ -1 (không có)


### Dạng tổng quát: tìm vị trí **đầu tiên** thỏa điều kiện

Rất nhiều bài không hỏi "có `x` không" mà hỏi "vị trí đầu tiên mà điều kiện `P(i)` đúng", với `P` **đơn điệu**: `False False … False True True … True`. Khung:

```text
lo, hi = 0, n            # đáp án nằm trong [lo, hi]; n nghĩa là "không có"
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid          # mid có thể là đáp án → giữ lại
    else:      lo = mid + 1      # mid chắc chắn không phải
return lo
```

Hai trường hợp quan trọng:

- **lower_bound**: vị trí đầu tiên `a[i] >= x` (`bisect_left`).
- **upper_bound**: vị trí đầu tiên `a[i] > x` (`bisect_right`).

Số lần xuất hiện của `x` = `upper_bound − lower_bound`.

In [2]:
from bisect import bisect_left, bisect_right

def dau_tien(n, P):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

b_bs = [1, 2, 2, 2, 5, 7]
lb = dau_tien(len(b_bs), lambda i: b_bs[i] >= 2)
ub = dau_tien(len(b_bs), lambda i: b_bs[i] > 2)
print("lower_bound(2) =", lb, "| upper_bound(2) =", ub, "| số lần xuất hiện:", ub - lb)
print("bisect_left / bisect_right:", bisect_left(b_bs, 2), bisect_right(b_bs, 2))
print("chèn 6 vào vị trí:", bisect_left(b_bs, 6))
print("số nguyên nhỏ nhất k với k*k >= 50:", dau_tien(100, lambda k: k * k >= 50))

lower_bound(2) = 1 | upper_bound(2) = 4 | số lần xuất hiện: 3
bisect_left / bisect_right: 1 4
chèn 6 vào vị trí: 5
số nguyên nhỏ nhất k với k*k >= 50: 8


### Lỗi hay gặp (nguồn lỗi số 1 là "lệch 1")

- Trộn hai kiểu khoảng: `while lo <= hi` đi với `hi = mid - 1`; `while lo < hi` đi với `hi = mid`. Trộn lẫn → **lặp vô hạn** hoặc bỏ sót.
- `lo = mid` (thay vì `mid + 1`) với `mid = (lo+hi)//2` → khi `hi = lo + 1` thì `mid = lo`, không tiến → lặp vô hạn.
- Áp dụng trên mảng **chưa sắp xếp** / điều kiện không đơn điệu.
- (Ngôn ngữ khác) `(lo + hi) / 2` tràn số → dùng `lo + (hi - lo) // 2`; Python không bị.

In [3]:
def sai_lo_bang_mid(a, x, gioi_han=20):
    lo, hi, buoc = 0, len(a) - 1, 0
    while lo < hi and buoc < gioi_han:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid            # SAI: không loại mid
        else:
            hi = mid
        buoc += 1
    return buoc
print("lo = mid: số vòng trước khi bị chặn:", sai_lo_bang_mid([1, 3], 3))

lo = mid: số vòng trước khi bị chặn: 20


## 2.2 Áp dụng vào bài này

Mảng đã sắp xếp nhưng bị xoay. Vẫn **tìm nhị phân** được: ở mỗi bước, ít nhất một nửa `[lo, mid]` hoặc `[mid, hi]` là **đã sắp xếp**; kiểm tra `target` có nằm trong nửa đó không để quyết định bỏ nửa nào.

# 3. Nền tảng: Binary Search trên mảng đã sắp xếp

Trên mảng tăng dần, so `target` với phần tử **giữa**:

- Bằng → tìm thấy.
- `target` nhỏ hơn → chỉ có thể nằm ở **nửa trái** → bỏ nửa phải.
- `target` lớn hơn → bỏ nửa trái.

Mỗi bước **loại một nửa**, nên sau `log₂ n` bước chỉ còn 1 phần tử. Với `n = 5000`, chỉ cần khoảng 13 bước.

Quy ước dùng trong notebook: đoạn tìm kiếm là `[lo, hi]` (**bao gồm cả hai đầu**), lặp khi `lo <= hi`.

In [4]:
from typing import List

def binary_search(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1          # bỏ nửa trái (kể cả mid)
        else:
            hi = mid - 1          # bỏ nửa phải (kể cả mid)
    return -1

arr = [0, 1, 2, 4, 5, 6, 7]
print(binary_search(arr, 5), binary_search(arr, 3), binary_search(arr, 0))

4 -1 0


**Vì sao `mid + 1` và `mid - 1`?** `mid` đã được kiểm tra (không bằng target), nên bỏ luôn nó. Nếu viết `lo = mid`, khi `lo == hi - 1` thì `mid == lo` mãi mãi → lặp vô hạn.

# 4. Cách dễ nghĩ nhất: duyệt tuyến tính

In [5]:
def search_linear(nums: List[int], target: int) -> int:
    for i, x in enumerate(nums):
        if x == target:
            return i
    return -1

print(search_linear([4, 5, 6, 7, 0, 1, 2], 0))

4


Đúng, nhưng `O(n)` — vi phạm yêu cầu `O(log n)`. Và nó bỏ phí thông tin "mảng gần như đã sắp xếp".

Binary search thường **không dùng trực tiếp được**: trên `[4, 5, 6, 7, 0, 1, 2]`, tìm `0`, `mid = 3` (giá trị 7). `0 < 7` → binary search thường bỏ nửa phải — nhưng `0` lại nằm ở nửa phải!

In [6]:
print("Binary search thường trên mảng xoay:", binary_search([4, 5, 6, 7, 0, 1, 2], 0), "(sai, đáp án là 4)")

Binary search thường trên mảng xoay: -1 (sai, đáp án là 4)


# 5. Quan sát then chốt: luôn có một nửa đã sắp xếp

Mảng xoay gồm **hai đoạn tăng dần** nối nhau: `[4, 5, 6, 7]` và `[0, 1, 2]`. Điểm "gãy" chỉ có **một**.

Cắt ở `mid` thành `[lo..mid]` và `[mid..hi]`: điểm gãy chỉ nằm ở **tối đa một** bên, nên **ít nhất một nửa là tăng dần thật sự**.

Làm sao biết nửa nào? So `nums[lo]` với `nums[mid]`:

- `nums[lo] <= nums[mid]` → **nửa trái** `[lo..mid]` tăng dần.
- Ngược lại → **nửa phải** `[mid..hi]` tăng dần.

Với nửa tăng dần, ta biết **chính xác khoảng giá trị** của nó (từ phần tử đầu tới phần tử cuối), nên kiểm tra được `target` có nằm trong đó không:

- Có → tìm tiếp trong nửa đó.
- Không → tìm ở nửa còn lại.

In [7]:
nums = [4, 5, 6, 7, 0, 1, 2]
for lo, hi in [(0, 6), (4, 6), (0, 3)]:
    mid = (lo + hi) // 2
    if nums[lo] <= nums[mid]:
        print(f"lo={lo}, mid={mid}, hi={hi}: nửa TRÁI {nums[lo:mid + 1]} tăng dần")
    else:
        print(f"lo={lo}, mid={mid}, hi={hi}: nửa PHẢI {nums[mid:hi + 1]} tăng dần")

lo=0, mid=3, hi=6: nửa TRÁI [4, 5, 6, 7] tăng dần
lo=4, mid=5, hi=6: nửa TRÁI [0, 1] tăng dần
lo=0, mid=1, hi=3: nửa TRÁI [4, 5] tăng dần


**Vì sao dùng `<=` chứ không phải `<`?** Khi `lo == mid` (đoạn còn 1–2 phần tử), nửa trái chỉ có **một** phần tử — luôn coi là tăng dần. Dùng `<` sẽ phân loại sai trường hợp này.

# 6. Lời giải chính

```text
while lo <= hi:
    mid = (lo + hi) // 2
    if nums[mid] == target: return mid
    if nums[lo] <= nums[mid]:                      # nửa trái tăng dần
        if nums[lo] <= target < nums[mid]: hi = mid - 1
        else:                              lo = mid + 1
    else:                                          # nửa phải tăng dần
        if nums[mid] < target <= nums[hi]: lo = mid + 1
        else:                              hi = mid - 1
```

In [8]:
def search_rotated(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid

        if nums[lo] <= nums[mid]:                       # nửa trái tăng dần
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                                           # nửa phải tăng dần
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1

print(search_rotated([4, 5, 6, 7, 0, 1, 2], 0))
print(search_rotated([4, 5, 6, 7, 0, 1, 2], 3))
print(search_rotated([1], 0))

4
-1
-1


`nums[lo] <= target < nums[mid]` là cú pháp **so sánh chuỗi** của Python, nghĩa là `nums[lo] <= target and target < nums[mid]`.

# 7. Mô phỏng từng bước

`nums = [4, 5, 6, 7, 0, 1, 2]`, `target = 0`.

| lo | hi | mid | nums[mid] | Nửa tăng dần | target trong đó? | Hành động |
|---|---|---|---|---|---|---|
| 0 | 6 | 3 | 7 | trái `[4..7]` | `4 <= 0 < 7`? không | `lo = 4` |
| 4 | 6 | 5 | 1 | trái `[0..1]` | `0 <= 0 < 1`? **có** | `hi = 4` |
| 4 | 4 | 4 | 0 | | | **tìm thấy, trả 4** |

In [9]:
def trace_search(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    print(f"nums = {nums}, target = {target}")
    while lo <= hi:
        mid = (lo + hi) // 2
        head = f"lo={lo} hi={hi} mid={mid} nums[mid]={nums[mid]:>2}"
        if nums[mid] == target:
            print(head, "-> TÌM THẤY")
            return mid
        if nums[lo] <= nums[mid]:
            inside = nums[lo] <= target < nums[mid]
            side = f"nửa trái {nums[lo:mid + 1]} tăng dần; target trong [{nums[lo]}, {nums[mid]})? {inside}"
            if inside:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            inside = nums[mid] < target <= nums[hi]
            side = f"nửa phải {nums[mid:hi + 1]} tăng dần; target trong ({nums[mid]}, {nums[hi]}]? {inside}"
            if inside:
                lo = mid + 1
            else:
                hi = mid - 1
        print(head, "|", side, f"-> lo={lo}, hi={hi}")
    print("Đoạn rỗng -> không có, trả -1")
    return -1

trace_search([4, 5, 6, 7, 0, 1, 2], 0)
print()
trace_search([4, 5, 6, 7, 0, 1, 2], 3)

nums = [4, 5, 6, 7, 0, 1, 2], target = 0
lo=0 hi=6 mid=3 nums[mid]= 7 | nửa trái [4, 5, 6, 7] tăng dần; target trong [4, 7)? False -> lo=4, hi=6
lo=4 hi=6 mid=5 nums[mid]= 1 | nửa trái [0, 1] tăng dần; target trong [0, 1)? True -> lo=4, hi=4
lo=4 hi=4 mid=4 nums[mid]= 0 -> TÌM THẤY

nums = [4, 5, 6, 7, 0, 1, 2], target = 3
lo=0 hi=6 mid=3 nums[mid]= 7 | nửa trái [4, 5, 6, 7] tăng dần; target trong [4, 7)? False -> lo=4, hi=6
lo=4 hi=6 mid=5 nums[mid]= 1 | nửa trái [0, 1] tăng dần; target trong [0, 1)? False -> lo=6, hi=6
lo=6 hi=6 mid=6 nums[mid]= 2 | nửa trái [2] tăng dần; target trong [2, 2)? False -> lo=7, hi=6
Đoạn rỗng -> không có, trả -1


-1

# 8. Cách 2: tìm điểm xoay rồi Binary Search thường

Điểm xoay = chỉ số của **phần tử nhỏ nhất** (bài **#153**). Tìm nó bằng binary search:

- Nếu `nums[mid] > nums[hi]` → điểm gãy nằm **bên phải** `mid` → `lo = mid + 1`.
- Ngược lại → phần tử nhỏ nhất ở `mid` hoặc bên trái → `hi = mid`.

(Ở đây dùng `lo < hi` và `hi = mid` — một biến thể binary search khác: đoạn thu hẹp tới khi còn đúng 1 phần tử.)

Sau đó `target` nằm trong `nums[:pivot]` hoặc `nums[pivot:]` — cả hai đều tăng dần.

In [10]:
def find_min_index(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
    return lo


def search_two_phase(nums: List[int], target: int) -> int:
    pivot = find_min_index(nums)
    n = len(nums)
    # chọn đoạn tăng dần chứa target
    if pivot > 0 and nums[0] <= target <= nums[pivot - 1]:
        lo, hi = 0, pivot - 1
    else:
        lo, hi = pivot, n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

print("điểm xoay:", find_min_index([4, 5, 6, 7, 0, 1, 2]))
print(search_two_phase([4, 5, 6, 7, 0, 1, 2], 0), search_two_phase([4, 5, 6, 7, 0, 1, 2], 5))

điểm xoay: 4
4 1


# 9. Phiên bản đúng format LeetCode

In [11]:
class Solution:
    def search(self, nums: List[int], target: int) -> int:
        lo, hi = 0, len(nums) - 1
        while lo <= hi:
            mid = (lo + hi) // 2
            if nums[mid] == target:
                return mid
            if nums[lo] <= nums[mid]:
                if nums[lo] <= target < nums[mid]:
                    hi = mid - 1
                else:
                    lo = mid + 1
            else:
                if nums[mid] < target <= nums[hi]:
                    lo = mid + 1
                else:
                    hi = mid - 1
        return -1

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.search([4, 5, 6, 7, 0, 1, 2], 0), 4)
check("Ví dụ 2", sol.search([4, 5, 6, 7, 0, 1, 2], 3), -1)
check("Ví dụ 3", sol.search([1], 0), -1)
check("Một phần tử, có", sol.search([1], 1), 0)
check("Không xoay", sol.search([1, 2, 3, 4, 5], 4), 3)
check("Hai phần tử xoay", sol.search([3, 1], 1), 1)
check("Target ở đầu", sol.search([5, 1, 3], 5), 0)
check("Target ở cuối", sol.search([6, 7, 1, 2, 3], 3), 4)

Ví dụ 1                        PASS  4
Ví dụ 2                        PASS  -1
Ví dụ 3                        PASS  -1
Một phần tử, có                PASS  0
Không xoay                     PASS  3
Hai phần tử xoay               PASS  1
Target ở đầu                   PASS  0
Target ở cuối                  PASS  4


## Kiểm tra mọi phép xoay của nhiều mảng

In [13]:
import random

random.seed(33)
count = 0
for _ in range(400):
    base = sorted(random.sample(range(-30, 30), random.randint(1, 15)))
    for k in range(len(base)):
        nums = base[k:] + base[:k]
        for target in range(-32, 32):
            expected = nums.index(target) if target in nums else -1
            assert sol.search(nums, target) == expected, (nums, target)
            assert search_two_phase(nums, target) == expected, (nums, target)
            count += 1
print(f"{count} truy vấn trên mọi phép xoay: hai cách đều đúng")

211008 truy vấn trên mọi phép xoay: hai cách đều đúng


# 10. Vì sao đúng? Độ phức tạp?

**Bất biến:** nếu `target` có trong mảng, chỉ số của nó luôn nằm trong `[lo, hi]`.

Ở mỗi bước, ta xác định một nửa **chắc chắn tăng dần** (mục 5). Trên nửa đó, `target` có mặt **khi và chỉ khi** nó nằm trong khoảng giá trị của nửa đó (vì mảng không có giá trị trùng). Do vậy:

- Nếu `target` thuộc khoảng → nó không thể ở nửa kia → bỏ nửa kia an toàn.
- Nếu không → nó không ở nửa này → bỏ nửa này an toàn.

Mỗi bước loại ít nhất một nửa (cộng `mid`) → đoạn rỗng sau `O(log n)` bước.

```text
Time:  O(log n)
Space: O(1)
```

# 11. Những lỗi dễ mắc

## Lỗi 1: Dùng `<` thay vì `<=` khi xác định nửa trái tăng dần

In [14]:
def wrong_strict(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] < nums[mid]:                    # SAI: thiếu dấu =
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1

print("Sai :", wrong_strict([3, 1], 1), "| Đúng:", sol.search([3, 1], 1))

Sai : -1 | Đúng: 1


Với `[3, 1]`: `lo = mid = 0`. Dùng `<` thì `3 < 3` sai → coi "nửa phải `[3, 1]` tăng dần" (thực ra không) → kiểm tra `3 < 1 <= 1` sai → `hi = -1` → bỏ lỡ.

## Lỗi 2: Đặt dấu bằng sai chỗ trong điều kiện khoảng

Nửa trái: `nums[lo] <= target < nums[mid]` — có `=` ở `lo` (target có thể là `nums[lo]`), không `=` ở `mid` (đã kiểm tra `mid`). Nửa phải tương tự: `nums[mid] < target <= nums[hi]`.

## Lỗi 3: So `target` với `nums[mid]` như binary search thường

Như mục 4, cho đáp án sai khi `target` nằm ở đoạn bên kia điểm gãy.

## Lỗi 4: Áp dụng nguyên si cho mảng **có giá trị trùng**

Với giá trị trùng (bài **#81**), `nums[lo] == nums[mid]` không cho biết nửa nào tăng dần (ví dụ `[1, 3, 1, 1, 1]`). Phải xử lý thêm (`lo += 1`), và độ phức tạp xấu nhất thành `O(n)`.

# 12. Mẹo nhận diện pattern

- Đề yêu cầu `O(log n)` trên mảng → gần như chắc chắn là **Binary Search**.
- Mảng "gần sắp xếp" (bị xoay, có đỉnh, …) → tìm **tính chất giúp loại một nửa** ở mỗi bước.
- Câu hỏi then chốt: *"Nhìn `nums[lo]`, `nums[mid]`, `nums[hi]`, mình biết chắc điều gì?"*

Bài liên quan: **#153 / #154 Find Minimum in Rotated Sorted Array**, **#81 Search in Rotated Sorted Array II**, **#162 Find Peak Element**, **#34 Find First and Last Position**.

## Template

```python
lo, hi = 0, n - 1
while lo <= hi:
    mid = (lo + hi) // 2
    if nums[mid] == target: return mid
    if nums[lo] <= nums[mid]:            # trái tăng dần
        if nums[lo] <= target < nums[mid]: hi = mid - 1
        else: lo = mid + 1
    else:                                 # phải tăng dần
        if nums[mid] < target <= nums[hi]: lo = mid + 1
        else: hi = mid - 1
return -1
```

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Trace `search` với `nums = [6, 7, 8, 1, 2, 3, 4, 5]` cho `target = 8` và `target = 5`.

## Bài 2 — Binary Search chuẩn

Viết lại binary search chuẩn (mảng tăng dần, không xoay).

In [15]:
def exercise_2_binary_search(nums: List[int], target: int) -> int:
    # TODO
    pass


# assert exercise_2_binary_search([1, 3, 5, 7], 5) == 2
# assert exercise_2_binary_search([1, 3, 5, 7], 4) == -1
# assert exercise_2_binary_search([], 1) == -1

## Bài 3 — Find Minimum in Rotated Sorted Array (LeetCode #153)

Trả về **giá trị nhỏ nhất** của mảng xoay (giá trị khác nhau), `O(log n)`.

```python
find_min([3, 4, 5, 1, 2])      -> 1
find_min([4, 5, 6, 7, 0, 1, 2]) -> 0
find_min([11, 13, 15, 17])     -> 11
```

In [16]:
def exercise_3_find_min(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_3_find_min([3, 4, 5, 1, 2]) == 1
# assert exercise_3_find_min([4, 5, 6, 7, 0, 1, 2]) == 0
# assert exercise_3_find_min([11, 13, 15, 17]) == 11

## Bài 4 — Điền chỗ trống

In [17]:
def exercise_4_search(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        # TODO 1: trả mid nếu tìm thấy
        # TODO 2: xác định nửa nào tăng dần
        # TODO 3: kiểm tra target có thuộc khoảng của nửa tăng dần không, cập nhật lo/hi
        pass
    return -1


# assert exercise_4_search([4, 5, 6, 7, 0, 1, 2], 0) == 4
# assert exercise_4_search([4, 5, 6, 7, 0, 1, 2], 3) == -1
# assert exercise_4_search([3, 1], 1) == 1

## Bài 5 — Mini challenge: có giá trị trùng (LeetCode #81)

Mảng xoay **có thể có giá trị trùng**. Trả về `True` nếu `target` có trong mảng.

```python
search_dup([2, 5, 6, 0, 0, 1, 2], 0) -> True
search_dup([2, 5, 6, 0, 0, 1, 2], 3) -> False
search_dup([1, 0, 1, 1, 1], 0)       -> True
```

In [18]:
def exercise_5_search_dup(nums: List[int], target: int) -> bool:
    # TODO
    pass


# assert exercise_5_search_dup([2, 5, 6, 0, 0, 1, 2], 0) is True
# assert exercise_5_search_dup([2, 5, 6, 0, 0, 1, 2], 3) is False
# assert exercise_5_search_dup([1, 0, 1, 1, 1], 0) is True

# 14. Gợi ý

**Bài 1**
- Mức 1: bước đầu `lo=0, hi=7, mid=3` (giá trị 1); nửa phải `[1..5]` tăng dần.

**Bài 3**
- Mức 1: so `nums[mid]` với `nums[hi]` (xem mục 8).
- Mức 2: `while lo < hi`, `hi = mid` (không phải `mid - 1`) vì `mid` có thể là phần tử nhỏ nhất.

**Bài 4**
- Mức 1: so với template mục 12.

**Bài 5**
- Mức 1: khi `nums[lo] == nums[mid] == nums[hi]`, không biết nửa nào tăng dần.
- Mức 2: khi đó chỉ cần `lo += 1; hi -= 1` (chắc chắn chúng không phải target vì bằng `nums[mid]`, đã kiểm tra).
- Mức 3: các trường hợp còn lại giống bài gốc.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [19]:
trace_search([6, 7, 8, 1, 2, 3, 4, 5], 8)
print()
trace_search([6, 7, 8, 1, 2, 3, 4, 5], 5)

nums = [6, 7, 8, 1, 2, 3, 4, 5], target = 8
lo=0 hi=7 mid=3 nums[mid]= 1 | nửa phải [1, 2, 3, 4, 5] tăng dần; target trong (1, 5]? False -> lo=0, hi=2
lo=0 hi=2 mid=1 nums[mid]= 7 | nửa trái [6, 7] tăng dần; target trong [6, 7)? False -> lo=2, hi=2
lo=2 hi=2 mid=2 nums[mid]= 8 -> TÌM THẤY

nums = [6, 7, 8, 1, 2, 3, 4, 5], target = 5
lo=0 hi=7 mid=3 nums[mid]= 1 | nửa phải [1, 2, 3, 4, 5] tăng dần; target trong (1, 5]? True -> lo=4, hi=7
lo=4 hi=7 mid=5 nums[mid]= 3 | nửa trái [2, 3] tăng dần; target trong [2, 3)? False -> lo=6, hi=7
lo=6 hi=7 mid=6 nums[mid]= 4 | nửa trái [4] tăng dần; target trong [4, 4)? False -> lo=7, hi=7
lo=7 hi=7 mid=7 nums[mid]= 5 -> TÌM THẤY


7

## Đáp án Bài 2

In [20]:
def answer_2_binary_search(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

assert answer_2_binary_search([1, 3, 5, 7], 5) == 2
assert answer_2_binary_search([1, 3, 5, 7], 4) == -1
assert answer_2_binary_search([], 1) == -1
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [21]:
def answer_3_find_min(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
    return nums[lo]

assert answer_3_find_min([3, 4, 5, 1, 2]) == 1
assert answer_3_find_min([4, 5, 6, 7, 0, 1, 2]) == 0
assert answer_3_find_min([11, 13, 15, 17]) == 11
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [22]:
def answer_4_search(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1

assert answer_4_search([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert answer_4_search([4, 5, 6, 7, 0, 1, 2], 3) == -1
assert answer_4_search([3, 1], 1) == 1
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [23]:
def answer_5_search_dup(nums: List[int], target: int) -> bool:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        if nums[lo] == nums[mid] == nums[hi]:
            lo += 1
            hi -= 1
        elif nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return False

assert answer_5_search_dup([2, 5, 6, 0, 0, 1, 2], 0) is True
assert answer_5_search_dup([2, 5, 6, 0, 0, 1, 2], 3) is False
assert answer_5_search_dup([1, 0, 1, 1, 1], 0) is True
for _ in range(3000):
    base = sorted(random.randint(0, 5) for _ in range(random.randint(1, 10)))
    k = random.randrange(len(base))
    nums = base[k:] + base[:k]
    t = random.randint(-1, 6)
    assert answer_5_search_dup(nums, t) == (t in nums), (nums, t)
print("Bài 5 PASS (kể cả 3000 test có trùng)")

Bài 5 PASS (kể cả 3000 test có trùng)


Lưu ý: với nhiều giá trị trùng (ví dụ `[1, 1, 1, ..., 1, 0, 1]`), mỗi bước có thể chỉ bỏ được 2 phần tử → xấu nhất `O(n)`. Đây là giới hạn cố hữu, không phải lỗi của lời giải.

# 16. Tổng kết

> Mảng xoay = hai đoạn tăng dần. Cắt ở `mid`, **ít nhất một nửa tăng dần** (nhận ra bằng `nums[lo] <= nums[mid]`). Kiểm tra `target` có nằm trong khoảng giá trị của nửa đó không để quyết định bỏ nửa nào.

---

## Checklist trước khi nộp

- [ ] `while lo <= hi`, cập nhật `mid ± 1`?
- [ ] Xác định nửa trái tăng dần bằng `nums[lo] <= nums[mid]` (có dấu `=`)?
- [ ] Khoảng nửa trái: `nums[lo] <= target < nums[mid]`; nửa phải: `nums[mid] < target <= nums[hi]`?
- [ ] Đã thử mảng 1 phần tử, 2 phần tử, không xoay, target ở hai đầu?